# ML-03 — Frame a Content-Decline Triage Task

**Chosen lane:** content-refresh triage. The goal is to help an editor decide which pseudonymized pages deserve attention first.

This notebook uses only the bundled, anonymized starter CSV. The decline label is a **proxy** derived from the starter data's trend rule; it is not an independent future outcome.

## 1. My lane as an ML task

**Task type: classification.** I would classify each content item as likely to be declining (`1`) or not declining (`0`) according to the starter-data proxy. This supports a triage decision: which pages should an editor inspect first?

The model output could later be converted into a ranked review queue by sorting pages by their predicted probability of decline.

## 2. Target or proxy

The target is `is_declining_proxy = 1` when `trend_direction == "down"`, otherwise `0`. This is a **defined proxy**, not an independently observed future outcome: `trend_direction` itself is calculated from the recent-versus-previous-30-day impression change.

For a production-quality experiment, I would prefer a forward-window outcome, such as whether impressions decline in the next 30 days. I must not use `trend_direction` or `trend_pct` as model features because they define this proxy.

## 3. Success metric and supported action

The primary metric is **recall**, reported alongside precision and the proxy base rate. Recall answers: of the pages labelled declining by this proxy, how many did the triage system identify? Precision answers: how much editor time is likely to be spent on pages in the flagged queue?

A useful operational version would also report precision at the review capacity (for example, Precision@50), because the editor can only inspect a finite number of pages. The output supports an editor choosing which content to audit or refresh first; it does not claim to predict Google's algorithm.

In [ ]:
from pathlib import Path
import pandas as pd

# Works in Colab and also when this notebook is run from the repository root.
local_path = Path("data/raw/content_refresh_anonymized.csv")
raw_url = (
    "https://raw.githubusercontent.com/Di-pesh/flyinterm/main/"
    "data/raw/content_refresh_anonymized.csv"
)
data_path = str(local_path) if local_path.exists() else raw_url
df = pd.read_csv(data_path)

print(f"Loaded {len(df):,} rows and {df.shape[1]} columns")
display(df.head(3))


## 4. The unit of analysis, as a real dataframe

I am using the `keyword article` slice as my lane. **One row = one pseudonymized content item/page**, with trailing-90-day activity and content metadata. `content_id` identifies the page for grouping and checking uniqueness; it is not a model feature.

In [ ]:
lane = df.loc[df["content_type"] == "keyword article"].copy()

assert lane["content_id"].is_unique, "Expected one row per content item in this starter slice"

# Keep the displayed slice small and readable while showing the unit and its signals.
display_columns = [
    "content_id", "client_id", "content_type", "main_intent",
    "impressions_90d", "clicks_90d", "sessions_90d",
    "avg_position", "days_since_last_update", "trend_direction",
]
display(lane[display_columns].head(10))
print(f"Lane rows: {len(lane):,}")
print(f"Unique pages: {lane['content_id'].nunique():,}")
print(f"Unique clients: {lane['client_id'].nunique():,}")


## 5. Sketch the target column

The next cell creates the proxy target and checks its prevalence. The source columns are retained only to explain and audit the label. They must be excluded from any later feature matrix.

In [ ]:
lane["is_declining_proxy"] = (lane["trend_direction"] == "down").astype("int8")

target_preview = lane[["content_id", "trend_direction", "trend_pct", "is_declining_proxy"]].head(10)
display(target_preview)

target_counts = lane["is_declining_proxy"].value_counts().sort_index()
target_summary = pd.DataFrame({
    "count": target_counts,
    "share": target_counts / len(lane),
}).rename(index={0: "not_declining", 1: "declining"})
display(target_summary)

assert set(lane["is_declining_proxy"].unique()).issubset({0, 1})
assert "trend_direction" not in display_columns or True  # source is visible for audit, not for modeling
print("Reminder: exclude trend_direction, trend_pct, and IDs from model features.")


## 6. Why ML beats a fixed rule here

A single rule such as `trend_pct < -20` merely reproduces the label definition. A useful triage model would combine several imperfect signals—traffic volume, position, engagement, freshness, content type, intent, and missingness—whose relationships may differ across clients. A model can learn a repeatable combination and produce a probability for prioritization, while a client-grouped validation split checks whether the pattern travels to unseen clients.

This is still decision-support: if the proxy is replaced by a genuine forward outcome, I would re-check the target window, leakage, class balance, and the editor's review capacity before making stronger claims.

## Self-check

- [x] Task type, target/proxy, metric, action, and ML rationale are stated.
- [x] The starter data is loaded and a lane slice is displayed.
- [x] The unit is explicit: one row is one content item/page.
- [x] The target column is sketched and its prevalence is shown.
- [ ] Before modeling: replace or clearly label this proxy, remove leakage columns and IDs, and validate by client.